# Optimización de hiperparámetros con KerasTuner

Vamos a utilizar el Dataset MNIST

In [2]:
import keras
from keras import layers
import keras_tuner
from keras_tuner.tuners import RandomSearch

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize the data
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten the images
x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

# Convert labels to one-hot encoding
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [1]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 11.5 MB/s eta 0:00:00


In [ ]:
print(x_train.shape, y_train.shape)

(60000, 784) (60000, 10)


Definimos el modelo. Vamos a definir los siguientes hiperparametros tuneables:

Número de capas de la red: Numero entero entre 1 y 3
Neuronas de cada capa: Número entero entre 32 y 512, en incrementos de 32
Tasa de entrenamiento: 0.001,0.0001,0,00001

In [3]:

# Define the model
def build_model(hp):
    model = keras.Sequential() # Input layer
    
    # Tune the number of layers
    for i in range(hp.Int('num_layers', 1, 3)): # Este bucle es para agregar entre 1 y 3 capas ocultas, dependiendo de lo que el tuner decida. El tuner es una herramienta que nos ayuda a encontrar los mejores hiperparámetros para nuestro modelo. En este caso, el número de capas ocultas es uno de esos hiperparámetros que el tuner va a ajustar durante la búsqueda.
        model.add(layers.Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=512, step=32),
            activation='relu'
        ))
    
    # Output layer
    model.add(layers.Dense(10, activation='softmax')) # Esto se hace para agregar la capa de salida con 10 unidades (una por cada clase) y una función de activación softmax, que es adecuada para problemas de clasificación multiclase como este. La función de activación softmax convierte las salidas del modelo en probabilidades, lo que facilita la interpretación de los resultados y la toma de decisiones basada en esas probabilidades.
    
    # Tune the learning rate
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Set up the tuner
tuner = RandomSearch( # El RandomSearch es una estrategia de búsqueda de hiperparámetros que selecciona aleatoriamente combinaciones de hiperparámetros para evaluar. A diferencia de GridSearch, que evalúa todas las combinaciones posibles, RandomSearch puede ser más eficiente en términos de tiempo y recursos, especialmente cuando el espacio de hiperparámetros es grande. Al usar RandomSearch, podemos encontrar buenas combinaciones de hiperparámetros sin tener que evaluar exhaustivamente todas las opciones, lo que puede ser especialmente útil cuando se trabaja con modelos complejos o grandes conjuntos de datos.
    build_model,
    objective='val_accuracy',
    max_trials=5, # esto son las ejecuciones que se van a realizar. Crea cinco modelos diferentes con diferentes combinaciones de hiperparámetros y los evalúa para encontrar el mejor. Cada modelo se entrena y evalúa en el conjunto de validación, y el objetivo es maximizar la precisión de validación (val_accuracy) para encontrar la mejor combinación de hiperparámetros.
    executions_per_trial=3, # los modelos los prueba 3 veces
    directory='my_dir',
    project_name='mnist_ann_tuning'
)

# Perform the search
tuner.search(x_train, y_train, epochs=5, validation_split=0.2)

# Retrieve the best model
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate the best model
loss, accuracy = best_model.evaluate(x_test, y_test)
print(f'Test Loss: {loss}')
print(f'Test Accuracy: {accuracy}')

# Summary of the best hyperparameters
best_hyperparameters = tuner.get_best_hyperparameters()[0]
print(best_hyperparameters.values)

Trial 5 Complete [00h 01m 02s]
val_accuracy: 0.9561388889948527

Best val_accuracy So Far: 0.9757777651151022
Total elapsed time: 00h 05m 22s


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9735 - loss: 0.0852
Test Loss: 0.07059051841497421
Test Accuracy: 0.9782000184059143
{'num_layers': 1, 'units_0': 448, 'learning_rate': 0.001, 'units_1': 64}
